In [32]:
import cv2
import face_recognition
import os
import serial
import time

# 建立主要的 facelook 資料夾
base_dir = "facelook"
if not os.path.exists(base_dir):
    os.makedirs(base_dir)
    print(f"📁 已自動建立主資料夾: {base_dir}")
else:
    print(f"📁 主資料夾 {base_dir} 已存在")

print("✅ Cell 1: 環境與套件載入完成")

📁 主資料夾 facelook 已存在
✅ Cell 1: 環境與套件載入完成


In [33]:
# 1. 建立 STM32 連線 (請確認 COM3 是正確的)
try:
    ser = serial.Serial('COM3', 115200, timeout=1)
    print("✅ STM32 序列埠連線成功")
except Exception as e:
    print(f"⚠️ 序列埠未連線，系統將進入無硬體模擬模式. 錯誤: {e}")
    ser = None

# 2. 設定 iPhone 鏡頭
video_url = "http://192.168.0.96:4747/video"
video_capture = cv2.VideoCapture(video_url)
print("📱 iPhone 鏡頭已就緒")

print("✅ Cell 2: 硬體連線初始化完成")

⚠️ 序列埠未連線，系統將進入無硬體模擬模式. 錯誤: could not open port 'COM3': FileNotFoundError(2, '系統找不到指定的檔案。', None, 2)
📱 iPhone 鏡頭已就緒
✅ Cell 2: 硬體連線初始化完成


In [34]:
# 置物櫃狀態常數
STATE_EMPTY = 0   # 空箱狀態 (可供註冊)
STATE_IN_USE = 1  # 滿箱狀態 (等待原物主取回)

# 全域狀態變數 (記憶體)
current_state = STATE_EMPTY
current_owner_encoding = None  # 暫存當下使用者的臉部特徵
current_owner_id = None        # 暫存當下使用者的編號
last_action_time = 0           # 紀錄動作時間，用作防呆冷卻

def get_next_id():
    """自動掃描 facelook 資料夾，尋找目前最大的數字並 +1"""
    max_id = 0
    for folder_name in os.listdir(base_dir):
        if folder_name.isdigit():
            num = int(folder_name)
            if num > max_id:
                max_id = num
    return max_id + 1

def trigger_door():
    """發送開門訊號給 STM32"""
    if ser:
        try:
            ser.write(b'1')
            print("⚡ 已發送開鎖指令給 STM32 (3秒後自動上鎖)。")
        except Exception as e:
            print(f"⚠️ 序列埠傳送失敗: {e}")
    else:
        print("⚡ (模擬) 開鎖指令已觸發，請想像門已經開了。")

print("✅ Cell 3: 狀態變數與輔助函式載入完成")

✅ Cell 3: 狀態變數與輔助函式載入完成


In [35]:
def process_store(frame, face_encodings):
    """處理『存物』邏輯：拍照建檔並鎖定權限"""
    global current_state, current_owner_encoding, current_owner_id, last_action_time

    if len(face_encodings) == 1:
        new_id = get_next_id()
        user_folder = os.path.join(base_dir, str(new_id))
        os.makedirs(user_folder, exist_ok=True)

        # 存下客人的照片留底
        photo_path = os.path.join(user_folder, "face.jpg")
        cv2.imwrite(photo_path, frame)

        # 將特徵寫入記憶體，切換成滿箱狀態
        current_owner_encoding = face_encodings[0]
        current_owner_id = new_id
        current_state = STATE_IN_USE
        last_action_time = time.time()

        print(f"📸 註冊成功！您是第 {new_id} 號客人。照片已存至 {user_folder}")
        trigger_door() # 開門讓客人放東西

    elif len(face_encodings) == 0:
        print("⚠️ 畫面中找不到人臉，請正對鏡頭！")
    else:
        print("⚠️ 畫面中有多張人臉，為確保安全，請確保只有一人入鏡！")

def process_retrieve(face_encodings):
    """處理『取物』邏輯：比對臉部並重置系統"""
    global current_state, current_owner_encoding, current_owner_id

    # 防呆：剛存入 5 秒內不允許立刻取物，避免誤觸
    if time.time() - last_action_time < 5:
        print("⏳ 剛存入物品，請稍候再取物...")
        return

    if len(face_encodings) > 0:
        match_found = False
        for face_encoding in face_encodings:
            match = face_recognition.compare_faces([current_owner_encoding], face_encoding, tolerance=0.45)
            if match[0]:
                match_found = True
                break

        if match_found:
            print(f"🔓 歡迎回來，第 {current_owner_id} 號客人！身分確認無誤。")
            trigger_door() # 開門讓客人拿走東西

            # 🌟 核心動作：取物完成，徹底清空記憶體，恢復空箱
            current_state = STATE_EMPTY
            current_owner_encoding = None
            current_owner_id = None
        else:
            print("❌ 臉部比對失敗：您不是這個置物櫃的主人！")
    else:
        print("⚠️ 請正對鏡頭進行取物辨識！")

print("✅ Cell 4: 存/取業務邏輯載入完成")

✅ Cell 4: 存/取業務邏輯載入完成


In [36]:
print("🎥 智慧置物櫃系統啟動！")
print("👉 系統操作說明：")
print("   [ s ] 註冊存物 (Store)")
print("   [ o ] 臉部解鎖取物 (Open)")
print("   [ q ] 關閉系統 (Quit)")

while True:
    ret, frame = video_capture.read()
    if not ret:
        print("❌ 影像讀取失敗，請檢查手機連線")
        break

    # 縮小畫面加速辨識
    small_frame = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)
    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    # 擷取畫面中的人臉特徵
    face_locations = face_recognition.face_locations(rgb_small_frame)
    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

    # 根據置物櫃狀態，改變畫面上的提示文字與顏色
    if current_state == STATE_EMPTY:
        status_text = "[ EMPTY ] Press 's' to Store"
        color = (0, 255, 0) # 綠色
    else:
        status_text = f"[ IN USE (User {current_owner_id}) ] Press 'o' to Open"
        color = (0, 0, 255) # 紅色

    cv2.putText(frame, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    # 幫畫面中的人臉畫框
    for (top, right, bottom, left) in face_locations:
        cv2.rectangle(frame, (left*2, top*2), (right*2, bottom*2), (255, 255, 0), 2)

    cv2.imshow('Smart Locker System', frame)

    # 監聽鍵盤按鍵
    key = cv2.waitKey(1) & 0xFF
    if key == ord('s') and current_state == STATE_EMPTY:
        process_store(frame, face_encodings)
    elif key == ord('o') and current_state == STATE_IN_USE:
        process_retrieve(face_encodings)
    elif key == ord('q'):
        break

# 釋放所有硬體資源
video_capture.release()
cv2.destroyAllWindows()
print("🛑 系統已安全關閉。")

🎥 智慧置物櫃系統啟動！
👉 系統操作說明：
   [ s ] 註冊存物 (Store)
   [ o ] 臉部解鎖取物 (Open)
   [ q ] 關閉系統 (Quit)
📸 註冊成功！您是第 3 號客人。照片已存至 facelook\3
⚡ (模擬) 開鎖指令已觸發，請想像門已經開了。
⏳ 剛存入物品，請稍候再取物...
⚠️ 請正對鏡頭進行取物辨識！
🔓 歡迎回來，第 3 號客人！身分確認無誤。
⚡ (模擬) 開鎖指令已觸發，請想像門已經開了。
📸 註冊成功！您是第 4 號客人。照片已存至 facelook\4
⚡ (模擬) 開鎖指令已觸發，請想像門已經開了。
🔓 歡迎回來，第 4 號客人！身分確認無誤。
⚡ (模擬) 開鎖指令已觸發，請想像門已經開了。
❌ 影像讀取失敗，請檢查手機連線
🛑 系統已安全關閉。
